In [ ]:
import nibabel as nib
import os
import numpy as np 
import matplotlib.pyplot as plt 
import pandas as pd 
import math 

import math
import os

import numpy as np
import pandas as pd

FLIST = [
    "/path/to/ABIDEII-NYU_1.csv",
    "/path/to/ABIDEII-SDSU_1.csv",
    "/path/to/ABIDEII-TCD_1.csv",
    "/path/to/ABIDEII-OHSU_1.csv",
    "/path/to/ABIDEII-KKI_1.csv",
]

VOLUME = "nu-to-mni305.mgz"
SEGMENTATION = "mri/aparc.DKTatlas+aseg.mgz"


def build_saliency_file_list(flist=FLIST):
    f, l, subj = [], [], []
    for college_csv in flist:
        csv = pd.read_csv(college_csv)
        csv = csv[["SUB_ID", "DX_GROUP", "AGE_AT_SCAN ", "SEX", "SRS_EDITION", "SRS_TOTAL_T"]]
        for i, r in csv.iterrows():
            sub = int(r[0])
            p = os.path.join("/path/tofreesurfer", str(sub), VOLUME)
            seg = os.path.join("/path/tofreesurfer", str(sub), SEGMENTATION)
            if not (os.path.exists(p) and os.path.exists(seg)):
                continue
            try:
                if math.isnan(r[5]):
                    continue
                if (r[2] <= 12) and (r[3] == 1):
                    if r[5] < 45:
                        f.append(p); l.append(0); subj.append(sub)
                    elif r[5] > 70:
                        f.append(p); l.append(1); subj.append(sub)
            except Exception as e:
                print(r[0], e)
    return np.array(f), np.array(l), np.array(subj)


f, l, subj = build_saliency_file_list()
low_subj = subj[l == 0]
high_subj = subj[l == 1]

sal_all = []
sal_low = []
sal_high = []

for sub in subj:
    sal_filename = '/path/to/ig_correct_fold/' + str(int(sub)) + '_saliency.npy'
    mask_filename = '/path/tofreesurfer/' + str(int(sub)) + '/mri/brainmask.mgz'
    mask = nib.load(mask_filename).get_fdata()
    sal = np.load(sal_filename)
    
    mask = (mask > 0).astype(float)
    sal = sal * mask
    
    
    sal_all.append(sal)
    if sub in low_subj:
        sal_low.append(sal)
    if sub in high_subj:
        sal_high.append(sal)
    
    print(subj.index(sub)/len(subj))
    

In [ ]:
all_avg = np.mean(sal_all, axis=0)
all_std = np.std(sal_all, axis=0)

low_avg = np.mean(sal_low, axis=0)
low_std = np.std(sal_low, axis=0)

high_avg = np.mean(sal_high, axis=0)
high_std = np.std(sal_high, axis=0)

In [ ]:
all_avg = nib.Nifti1Image(all_avg, np.eye(4))
all_std = nib.Nifti1Image(all_std, np.eye(4))

low_avg = nib.Nifti1Image(low_avg, np.eye(4))
low_std = nib.Nifti1Image(low_std, np.eye(4))

high_avg = nib.Nifti1Image(high_avg, np.eye(4))
high_std = nib.Nifti1Image(high_std, np.eye(4))

In [ ]:
mni305_to_mni152 = np.array([
    [0.9975, -0.0073,  0.0176, -0.0429],
    [0.0146,  1.0009, -0.0024,  1.5496],
    [-0.0130, -0.0093,  0.9971,  1.1840],
    [0,        0,       0,       1]
])

ref = nib.load('/path/to/freesurfer/29173/mri/orig.mgz')

def to_mni152(img, ref):
    new_img = nib.Nifti1Image(img.get_fdata(), ref.affine)
    new_affine = mni305_to_mni152 @ ref.affine
    return nib.Nifti1Image(img.get_fdata(), new_affine)

all_avg  = to_mni152(all_avg, ref)
all_std  = to_mni152(all_std, ref)
low_avg  = to_mni152(low_avg, ref)
low_std  = to_mni152(low_std, ref)
high_avg = to_mni152(high_avg, ref)
high_std = to_mni152(high_std, ref)

In [ ]:
nib.save(all_avg,'/path/to/sal_all_avg.nii')
nib.save(all_std,'/path/to/sal_all_std.nii')

nib.save(low_avg,'/path/to/sal_low_avg.nii')
nib.save(low_std,'/path/to/sal_low_std.nii')

nib.save(high_avg,'/path/to/sal_high_avg.nii')
nib.save(high_std,'/path/to/sal_high_std.nii')

In [ ]:
import numpy as np
import siibra
from nilearn.plotting import plot_img_on_surf
from nilearn import image
import matplotlib.pyplot as plt
import nibabel as nib

In [ ]:
avg_all = nib.load('/path/to/sal_all_avg.nii')
std_all = nib.load('/path/to/sal_all_std.nii')

avg_low = nib.load('/path/to/sal_low_avg.nii')
std_low = nib.load('/path/to/sal_low_std.nii')

avg_high = nib.load('/path/to/sal_high_avg.nii')
std_high = nib.load('/path/to/sal_high_std.nii')

In [ ]:
mni305_to_mni152 = np.array([
    [0.9975, -0.0073,  0.0176, -0.0429],
    [0.0146,  1.0009, -0.0024,  1.5496],
    [-0.0130, -0.0093,  0.9971,  1.1840],
    [0,        0,       0,       1]
])

ref = nib.load('/path/to/freesurfer/29173/mri/orig.mgz')

def to_mni152(img, ref):
    new_img = nib.Nifti1Image(img.get_fdata(), ref.affine)
    new_affine = mni305_to_mni152 @ ref.affine
    return nib.Nifti1Image(img.get_fdata(), new_affine)

avg_all  = to_mni152(avg_all, ref)
std_all  = to_mni152(std_all, ref)
avg_low  = to_mni152(avg_low, ref)
std_low  = to_mni152(std_low, ref)
avg_high = to_mni152(avg_high, ref)
std_high = to_mni152(std_high, ref)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

vmin = -2.5e-06
vmax =  1e-06

zero_pos = (0 - vmin) / (vmax - vmin)

base_cmap = plt.get_cmap("RdBu_r")

n = 256

left_positions = np.linspace(0, zero_pos, n // 2)
right_positions = np.linspace(zero_pos, 1, n // 2)

left_colors = base_cmap(np.linspace(0, 0.5, n // 2))
right_colors = base_cmap(np.linspace(0.5, 1, n // 2))

positions = np.concatenate([left_positions, right_positions])
colors = np.vstack([left_colors, right_colors])

shifted_cmap = LinearSegmentedColormap.from_list(
    "shifted_RdBu",
    list(zip(positions, colors))
)

In [ ]:
plt.figure(figsize=(16, 12), dpi=150)
fig1 = plot_img_on_surf(
    stat_map=avg_all,
    views=["lateral", "medial"],
    hemispheres=["left", "right"],
    title="avgall",
    bg_on_data=True,
    cmap=shifted_cmap,
    colorbar=True,
    threshold=None,
    vmin=vmin,
    vmax=vmax,
    symmetric_cbar=False
)
plt.show()

In [ ]:
plt.figure(figsize=(16, 12), dpi=150)
fig1 = plot_img_on_surf(
    stat_map=avg_high,
    views=["lateral", "medial"],
    hemispheres=["left", "right"],
    bg_on_data=True,
    title="avghigh",
    cmap=shifted_cmap,
    colorbar=True,
    threshold=None,
    vmin=vmin,
    vmax=vmax,
    symmetric_cbar=False
)
plt.show()

In [ ]:
plt.figure(figsize=(16, 12), dpi=150)
fig1 = plot_img_on_surf(
    stat_map=avg_low,
    views=["lateral", "medial"],
    hemispheres=["left", "right"],
    bg_on_data=True,
    title="avglow",
    cmap=shifted_cmap,
    colorbar=True,
    threshold=None,
    vmin=vmin,
    vmax=vmax,
    symmetric_cbar=False
)
plt.show()

In [ ]:
plt.figure(figsize=(16, 12), dpi=150)
fig1 = plot_img_on_surf(
    stat_map=std_all,
    views=["lateral", "medial"],
    hemispheres=["left", "right"],
    bg_on_data=True,
    title="stdall",
    cmap='pink_r',
    colorbar=True,
    threshold=None,
    vmax=0.000005
)
plt.show()

In [ ]:
plt.figure(figsize=(16, 12), dpi=150)
fig1 = plot_img_on_surf(
    stat_map=std_high,
    views=["lateral", "medial"],
    hemispheres=["left", "right"],
    bg_on_data=True,
    title="stdhigh",
    cmap='pink_r',
    colorbar=True,
    threshold=None,
    vmax=0.000005
)
plt.show()

In [ ]:
plt.figure(figsize=(16, 12), dpi=150)
fig1 = plot_img_on_surf(
    stat_map=std_low,
    views=["lateral", "medial"],
    hemispheres=["left", "right"],
    bg_on_data=True,
    title="stdlow",
    cmap='pink_r',
    colorbar=True,
    threshold=None,
    vmax=0.000005
)
plt.show()